In [1]:
# Portfolio Project - Module 8
# April M. Vuylsteke
# Colorado State University Global
# MIS505: Data Wrangling
# Dr. Emmanuel Tsukerman
# November 8, 2026

In [2]:
# Step 1: import libraries
import pandas as pd
import numpy as np
import sqlite3
import json

In [3]:
# Step 2: read ed data
data_education_raw = pd.read_csv(
    r"C:\Users\AprilVuylsteke\OneDrive - Atmosera\Documents\My Projects\z. MIS505 Data Wrangling\Module 8\Portfolio Project\Data\API_SE.PRM.ENRL_DS2_en_csv_v2_4538564.csv",
    skiprows=4
)

# Check first 5 rows, dimensions
print(data_education_raw.head())
print(data_education_raw.shape)

                  Country Name Country Code             Indicator Name  \
0                        Aruba          ABW  Primary education, pupils   
1  Africa Eastern and Southern          AFE  Primary education, pupils   
2                  Afghanistan          AFG  Primary education, pupils   
3   Africa Western and Central          AFW  Primary education, pupils   
4                       Angola          AGO  Primary education, pupils   

  Indicator Code  1960  1961  1962  1963  1964  1965  ...       2013  \
0    SE.PRM.ENRL   NaN   NaN   NaN   NaN   NaN   NaN  ...     9803.0   
1    SE.PRM.ENRL   NaN   NaN   NaN   NaN   NaN   NaN  ...        NaN   
2    SE.PRM.ENRL   NaN   NaN   NaN   NaN   NaN   NaN  ...  5986268.0   
3    SE.PRM.ENRL   NaN   NaN   NaN   NaN   NaN   NaN  ...        NaN   
4    SE.PRM.ENRL   NaN   NaN   NaN   NaN   NaN   NaN  ...        NaN   

        2014       2015       2016       2017       2018       2019  2020  \
0     9816.0        NaN        NaN        NaN

In [4]:
# Step 3: Look at education data
print(data_education_raw.columns)
print(data_education_raw.info())

Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968',
       '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977',
       '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986',
       '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995',
       '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004',
       '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013',
       '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021',
       'Unnamed: 66'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 67 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    266 non-null    str    
 1   Country Code    266 non-null    str    
 2   Indicator Name  266 non-null    str    
 3   Indicator Code

In [5]:
# Step 4: Filter for Belgium
data_education_belgium = data_education_raw[
    (data_education_raw["Country Name"] == "Belgium") &
    (data_education_raw["Indicator Code"] == "SE.PRM.ENRL")
]

print(data_education_belgium)

   Country Name Country Code             Indicator Name Indicator Code  1960  \
17      Belgium          BEL  Primary education, pupils    SE.PRM.ENRL   NaN   

    1961  1962  1963  1964  1965  ...      2013      2014      2015      2016  \
17   NaN   NaN   NaN   NaN   NaN  ...  764137.0  773568.0  782606.0  797390.0   

        2017      2018      2019  2020  2021  Unnamed: 66  
17  814974.0  821236.0  823109.0   NaN   NaN          NaN  

[1 rows x 67 columns]


In [6]:
# Step 5: Get years, enrollment values
years = list(range(2001, 2021))
primary_enrollment_values = []

for year in years: primary_enrollment_values.append(data_education_belgium[str(year)].values[0])

In [7]:
# Step 6: Create enrollment df
data_primary_enrollment = pd.DataFrame({
    "Year": years,
    "Primary Enrollment": primary_enrollment_values
})

print(data_primary_enrollment)

    Year  Primary Enrollment
0   2001            771889.0
1   2002            767787.0
2   2003            761730.0
3   2004            747111.0
4   2005            738580.0
5   2006            732808.0
6   2007            732411.0
7   2008            733052.0
8   2009            731603.0
9   2010            731761.0
10  2011            736062.0
11  2012            744331.0
12  2013            764137.0
13  2014            773568.0
14  2015            782606.0
15  2016            797390.0
16  2017            814974.0
17  2018            821236.0
18  2019            823109.0
19  2020                 NaN


In [8]:
# Step 7: Find missing enrollment values
print(data_primary_enrollment.isnull().sum())
print(data_primary_enrollment[
    data_primary_enrollment["Primary Enrollment"].isnull()
])

Year                  0
Primary Enrollment    1
dtype: int64
    Year  Primary Enrollment
19  2020                 NaN


In [9]:
# Step 8: Interpolate missing values
data_primary_enrollment.loc[data_primary_enrollment["Year"] < 2020,"Primary Enrollment"] = \
    data_primary_enrollment.loc[data_primary_enrollment["Year"] < 2020,"Primary Enrollment"].interpolate()

# Check remaining missing values
print(data_primary_enrollment.isnull().sum())
print(data_primary_enrollment)

Year                  0
Primary Enrollment    1
dtype: int64
    Year  Primary Enrollment
0   2001            771889.0
1   2002            767787.0
2   2003            761730.0
3   2004            747111.0
4   2005            738580.0
5   2006            732808.0
6   2007            732411.0
7   2008            733052.0
8   2009            731603.0
9   2010            731761.0
10  2011            736062.0
11  2012            744331.0
12  2013            764137.0
13  2014            773568.0
14  2015            782606.0
15  2016            797390.0
16  2017            814974.0
17  2018            821236.0
18  2019            823109.0
19  2020                 NaN


In [10]:
# Step 9: Read GDP data
data_gdp_raw = pd.read_csv(
    r"C:\Users\AprilVuylsteke\OneDrive - Atmosera\Documents\My Projects\z. MIS505 Data Wrangling\Module 8\Portfolio Project\Data\WDIData.csv"
)

print(data_gdp_raw.head())
print(data_gdp_raw.shape)

                  Country Name Country Code  \
0  Africa Eastern and Southern          AFE   
1  Africa Eastern and Southern          AFE   
2  Africa Eastern and Southern          AFE   
3  Africa Eastern and Southern          AFE   
4  Africa Eastern and Southern          AFE   

                                      Indicator Name     Indicator Code  1960  \
0  Access to clean fuels and technologies for coo...     EG.CFT.ACCS.ZS   NaN   
1  Access to clean fuels and technologies for coo...  EG.CFT.ACCS.RU.ZS   NaN   
2  Access to clean fuels and technologies for coo...  EG.CFT.ACCS.UR.ZS   NaN   
3            Access to electricity (% of population)     EG.ELC.ACCS.ZS   NaN   
4  Access to electricity, rural (% of rural popul...  EG.ELC.ACCS.RU.ZS   NaN   

   1961  1962  1963  1964  1965  ...       2013       2014       2015  \
0   NaN   NaN   NaN   NaN   NaN  ...  16.936004  17.337896  17.687093   
1   NaN   NaN   NaN   NaN   NaN  ...   6.499471   6.680066   6.859110   
2   NaN   N

In [11]:
# Step 10: Look at GDP data
print(data_gdp_raw.columns)
print(data_gdp_raw.info())

Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968',
       '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977',
       '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986',
       '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995',
       '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004',
       '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013',
       '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021',
       'Unnamed: 66'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 383572 entries, 0 to 383571
Data columns (total 67 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Country Name    383572 non-null  str    
 1   Country Code    383572 non-null  str    
 2   Indicator Name  383572 non-null  str    
 3   Ind

In [12]:
# Step 11: Get Belgium's GDP per capita
data_gdp_belgium = data_gdp_raw[(data_gdp_raw["Country Name"] == "Belgium") & (data_gdp_raw["Indicator Code"] == "NY.GDP.PCAP.CD")]

print(data_gdp_belgium)

      Country Name Country Code                Indicator Name  Indicator Code  \
97088      Belgium          BEL  GDP per capita (current US$)  NY.GDP.PCAP.CD   

              1960         1961         1962         1963         1964  \
97088  1273.691659  1350.197673  1438.523233  1535.023729  1701.846276   

              1965  ...          2013          2014          2015  \
97088  1835.594766  ...  46757.951856  47764.071512  41008.296719   

               2016          2017          2018          2019        2020  \
97088  42012.622719  44198.482391  47549.208049  46599.111335  45189.3669   

               2021  Unnamed: 66  
97088  51767.788572          NaN  

[1 rows x 67 columns]


In [13]:
# Step 12: Get GDP per capita values
gdp_per_capita_values = []

for year in years: gdp_per_capita_values.append(data_gdp_belgium[str(year)].values[0])

In [14]:
# Step 13: Create df for GDP per capita
data_gdp_per_capita = pd.DataFrame({
    "Year": years,
    "GDP": gdp_per_capita_values
})

print(data_gdp_per_capita)

    Year           GDP
0   2001  23015.071263
1   2002  25006.191397
2   2003  30655.209268
3   2004  35429.407793
4   2005  36809.701340
5   2006  38705.106796
6   2007  44319.165449
7   2008  48303.397956
8   2009  44760.291244
9   2010  44184.946354
10  2011  47410.566928
11  2012  44670.560685
12  2013  46757.951856
13  2014  47764.071512
14  2015  41008.296719
15  2016  42012.622719
16  2017  44198.482391
17  2018  47549.208049
18  2019  46599.111335
19  2020  45189.366900


In [15]:
# Step 14: Find missing GDP values
print(data_gdp_per_capita.isnull().sum())

Year    0
GDP     0
dtype: int64


In [16]:
# Step 15: Reset indexes to merge
data_primary_enrollment.reset_index(drop=True,inplace=True)

data_gdp_per_capita.reset_index(drop=True,inplace=True)

In [17]:
# Step 16: Merge enrollment, GDP by Year
data_education_gdp_final = pd.merge(data_primary_enrollment, data_gdp_per_capita, on="Year")

print(data_education_gdp_final)

    Year  Primary Enrollment           GDP
0   2001            771889.0  23015.071263
1   2002            767787.0  25006.191397
2   2003            761730.0  30655.209268
3   2004            747111.0  35429.407793
4   2005            738580.0  36809.701340
5   2006            732808.0  38705.106796
6   2007            732411.0  44319.165449
7   2008            733052.0  48303.397956
8   2009            731603.0  44760.291244
9   2010            731761.0  44184.946354
10  2011            736062.0  47410.566928
11  2012            744331.0  44670.560685
12  2013            764137.0  46757.951856
13  2014            773568.0  47764.071512
14  2015            782606.0  41008.296719
15  2016            797390.0  42012.622719
16  2017            814974.0  44198.482391
17  2018            821236.0  47549.208049
18  2019            823109.0  46599.111335
19  2020                 NaN  45189.366900


In [18]:
# Step 17: Verify combined data
print(data_education_gdp_final.shape)
print(data_education_gdp_final.isnull().sum())
print(data_education_gdp_final.dtypes)
print(data_education_gdp_final)

(20, 3)
Year                  0
Primary Enrollment    1
GDP                   0
dtype: int64
Year                    int64
Primary Enrollment    float64
GDP                   float64
dtype: object
    Year  Primary Enrollment           GDP
0   2001            771889.0  23015.071263
1   2002            767787.0  25006.191397
2   2003            761730.0  30655.209268
3   2004            747111.0  35429.407793
4   2005            738580.0  36809.701340
5   2006            732808.0  38705.106796
6   2007            732411.0  44319.165449
7   2008            733052.0  48303.397956
8   2009            731603.0  44760.291244
9   2010            731761.0  44184.946354
10  2011            736062.0  47410.566928
11  2012            744331.0  44670.560685
12  2013            764137.0  46757.951856
13  2014            773568.0  47764.071512
14  2015            782606.0  41008.296719
15  2016            797390.0  42012.622719
16  2017            814974.0  44198.482391
17  2018            821236.0 

In [19]:
# Step 18: Export final dataset to CSV
data_education_gdp_final.to_csv(
   r"C:\Users\AprilVuylsteke\OneDrive - Atmosera\Documents\My Projects\z. MIS505 Data Wrangling\Module 8\Portfolio Project\Data\Belgium_Education_GDP.csv",
    index=False
)

data_education_gdp_csv = pd.read_csv(
    r"C:\Users\AprilVuylsteke\OneDrive - Atmosera\Documents\My Projects\z. MIS505 Data Wrangling\Module 8\Portfolio Project\Data\Belgium_Education_GDP.csv"
)

print(data_education_gdp_csv)

    Year  Primary Enrollment           GDP
0   2001            771889.0  23015.071263
1   2002            767787.0  25006.191397
2   2003            761730.0  30655.209268
3   2004            747111.0  35429.407793
4   2005            738580.0  36809.701340
5   2006            732808.0  38705.106796
6   2007            732411.0  44319.165449
7   2008            733052.0  48303.397956
8   2009            731603.0  44760.291244
9   2010            731761.0  44184.946354
10  2011            736062.0  47410.566928
11  2012            744331.0  44670.560685
12  2013            764137.0  46757.951856
13  2014            773568.0  47764.071512
14  2015            782606.0  41008.296719
15  2016            797390.0  42012.622719
16  2017            814974.0  44198.482391
17  2018            821236.0  47549.208049
18  2019            823109.0  46599.111335
19  2020                 NaN  45189.366900


In [20]:
# Step 19: Connect to SQLite db
conn_education_gdp = sqlite3.connect(
    r"C:\Users\AprilVuylsteke\OneDrive - Atmosera\Documents\My Projects\z. MIS505 Data Wrangling\Module 8\Portfolio Project\Data\Belgium_Education_GDP.db"
)

cursor_education_gdp = conn_education_gdp.cursor()
cursor_education_gdp.execute("PRAGMA foreign_keys = 1")

In [21]:
# Step 20: Check if education_gdp table already exists (if running code multiple times to correct issues, it may)
cursor_education_gdp.execute("DROP TABLE IF EXISTS education_gdp;")

sql = "CREATE TABLE education_gdp \
    (Year integer PRIMARY KEY, \
    Primary_Enrollment real, \
    GDP real)"

cursor_education_gdp.execute(sql)
conn_education_gdp.commit()

In [22]:
# Step 21: Print table names in db, check structure
rows = cursor_education_gdp.execute("SELECT name \
    FROM sqlite_master \
    WHERE type='table';")

for row in rows: print(row)

rows = cursor_education_gdp.execute("PRAGMA table_info(education_gdp);")

for row in rows: print(row)

('education_gdp',)
(0, 'Year', 'INTEGER', 0, None, 1)
(1, 'Primary_Enrollment', 'REAL', 0, None, 0)
(2, 'GDP', 'REAL', 0, None, 0)


In [23]:
# Step 22: Insert enrollment, GDP records
for i in range(len(data_education_gdp_final)):
    row = data_education_gdp_final.loc[i]

    cursor_education_gdp.execute("INSERT INTO education_gdp \
        VALUES (?, ?, ?);", tuple(row))

conn_education_gdp.commit()

In [24]:
# Step 23: Print db records
rows = cursor_education_gdp.execute("SELECT * \
    FROM education_gdp \
    ORDER BY Year ASC;")

for row in rows: print(row)

(2001, 771889.0, 23015.0712632462)
(2002, 767787.0, 25006.191397109)
(2003, 761730.0, 30655.2092679024)
(2004, 747111.0, 35429.4077933344)
(2005, 738580.0, 36809.7013403619)
(2006, 732808.0, 38705.1067959147)
(2007, 732411.0, 44319.165448813)
(2008, 733052.0, 48303.397956286)
(2009, 731603.0, 44760.2912443709)
(2010, 731761.0, 44184.946353964)
(2011, 736062.0, 47410.5669277464)
(2012, 744331.0, 44670.5606845101)
(2013, 764137.0, 46757.9518559598)
(2014, 773568.0, 47764.0715120833)
(2015, 782606.0, 41008.296719472)
(2016, 797390.0, 42012.6227191016)
(2017, 814974.0, 44198.4823908691)
(2018, 821236.0, 47549.2080494546)
(2019, 823109.0, 46599.1113350938)
(2020, None, 45189.3669003126)


In [25]:
# Step 24: Print num of records
rows = cursor_education_gdp.execute("SELECT COUNT(*) \
    FROM education_gdp;")

for row in rows: print(row)

(20,)


In [26]:
# Step 25: Close db connection
conn_education_gdp.close()

In [27]:
# Step 26: Create JSON data desc
data_description = {
    "dataset": "Belgium_Education_GDP.csv",
    "country": "Belgium",
    "description": "Primary enrollment and GDP per capita, 2001-2020",
    "missing_data": "Primary enrollment for 2020 not available in data",
    "attributes": [
        {
            "column_name": "Year",
            "data_type": "integer",
            "meaning": "Calendar year of observation"
        },
        {
            "column_name": "Primary Enrollment",
            "data_type": "float",
            "meaning": "Num pupils enrolled in primary education"
        },
        {
            "column_name": "GDP",
            "data_type": "float",
            "meaning": "GDP per capita in current USD"
        }
    ]
}

print(data_description)

{'dataset': 'Belgium_Education_GDP.csv', 'country': 'Belgium', 'description': 'Primary enrollment and GDP per capita, 2001-2020', 'missing_data': 'Primary enrollment for 2020 not available in data', 'attributes': [{'column_name': 'Year', 'data_type': 'integer', 'meaning': 'Calendar year of observation'}, {'column_name': 'Primary Enrollment', 'data_type': 'float', 'meaning': 'Num pupils enrolled in primary education'}, {'column_name': 'GDP', 'data_type': 'float', 'meaning': 'GDP per capita in current USD'}]}


In [28]:
# Step 27: Export JSON desc
file_output = open(r"C:\Users\AprilVuylsteke\OneDrive - Atmosera\Documents\My Projects\z. MIS505 Data Wrangling\Module 8\Portfolio Project\Data\Belgium_Education_GDP_description.json", "w")

json.dump(data_description, file_output, indent=4)

file_output.close()

In [29]:
# Step 28: Read JSON desc
file_input = open(r"C:\Users\AprilVuylsteke\OneDrive - Atmosera\Documents\My Projects\z. MIS505 Data Wrangling\Module 8\Portfolio Project\Data\Belgium_Education_GDP_description.json", "r")

data_description_check = json.load(file_input)

file_input.close()

print(data_description_check)

{'dataset': 'Belgium_Education_GDP.csv', 'country': 'Belgium', 'description': 'Primary enrollment and GDP per capita, 2001-2020', 'missing_data': 'Primary enrollment for 2020 not available in data', 'attributes': [{'column_name': 'Year', 'data_type': 'integer', 'meaning': 'Calendar year of observation'}, {'column_name': 'Primary Enrollment', 'data_type': 'float', 'meaning': 'Num pupils enrolled in primary education'}, {'column_name': 'GDP', 'data_type': 'float', 'meaning': 'GDP per capita in current USD'}]}
